# VisDrone Fine-tune — yolo11l
Fine-tunes yolo11l on VisDrone2019-DET (drone-view traffic).  
**Runtime → Change runtime type → T4 GPU** before running.

Result: `yolo11l-visdrone-ft.pt` saved to your Google Drive.

In [ ]:
# 1. Mount Google Drive (to save the trained model)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install ultralytics
!pip install ultralytics -q
import ultralytics
print('ultralytics', ultralytics.__version__)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!')

In [ ]:
# 3. Train — VisDrone dataset auto-downloads (~2.3GB)
from ultralytics import YOLO
from pathlib import Path

model = YOLO('yolo11l.pt')   # auto-downloads from ultralytics

results = model.train(
    data='VisDrone.yaml',
    epochs=50,
    imgsz=640,
    batch=16,          # T4 has 15GB, batch=16 fits fine
    device=0,          # GPU
    project='/content/runs',
    name='yolo11l-visdrone-ft',
    exist_ok=True,

    # fine-tune settings
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    freeze=10,          # freeze backbone
    patience=15,

    # augmentation
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, flipud=0.0,
    mosaic=0.5,

    save=True,
    plots=True,
)

In [ ]:
# 4. Copy best model to Google Drive
import shutil

best = Path('/content/runs/yolo11l-visdrone-ft/weights/best.pt')
dest_dir = Path('/content/drive/MyDrive/trafficlab_models')
dest_dir.mkdir(parents=True, exist_ok=True)
dest = dest_dir / 'yolo11l-visdrone-ft.pt'

shutil.copy(best, dest)
print(f'Saved to Google Drive: {dest}')
print(f'Model size: {dest.stat().st_size / 1e6:.1f} MB')

In [ ]:
# 5. (Optional) Show training results
from IPython.display import Image
Image('/content/runs/yolo11l-visdrone-ft/results.png')